In [ ]:
                                                                             
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings("ignore")

                                                                              
TRAIN_PATH  = "train.csv"
TEST_PATH   = "test.csv"
OUTDIR      = "results_titanic"
DEPTHS      = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]                   
RANDOM_STATE = 42
os.makedirs(OUTDIR, exist_ok=True); plt.rcParams["figure.dpi"] = 110

                                                                              
df = pd.read_csv(TRAIN_PATH)
print("train.csv shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution (Survived):\n", df["Survived"].value_counts().to_string())

                                                                              
def preprocess(data):
    d = data.copy()
    d = d.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")
    d["Age"]      = d["Age"].fillna(d["Age"].median())
    d["Fare"]     = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna(d["Embarked"].mode()[0])
    d["Sex"]      = d["Sex"].map({"male": 0, "female": 1})
    d["Embarked"] = d["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    return d

data = preprocess(df)
X = data.drop(columns=["Survived"])
y = data["Survived"]
FEATURES = list(X.columns)
print("\nFeatures used:", FEATURES)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

                                                                              
                                                
                                                                              
print("\n" + "="*70)
print("PART A - DECISION TREE (varying max_depth)")
print("="*70)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows, trees = [], {}
for dep in DEPTHS:
    clf = DecisionTreeClassifier(max_depth=dep, random_state=RANDOM_STATE).fit(X_train, y_train)
    tr_pred, te_pred = clf.predict(X_train), clf.predict(X_test)
    cvs = cross_val_score(DecisionTreeClassifier(max_depth=dep, random_state=RANDOM_STATE),
                          X, y, cv=cv, scoring="accuracy")
    rows.append({
        "max_depth": "None" if dep is None else dep,
        "actual_depth": clf.get_depth(),
        "leaves": clf.get_n_leaves(),
        "train_acc": accuracy_score(y_train, tr_pred),
        "test_acc":  accuracy_score(y_test,  te_pred),
        "cv_acc":    cvs.mean(),
        "cv_std":    cvs.std(),
        "precision": precision_score(y_test, te_pred),
        "recall":    recall_score(y_test, te_pred),
        "f1":        f1_score(y_test, te_pred)})
    trees[dep] = clf
    print(f"depth={str(dep):<5} train={rows[-1]['train_acc']:.4f} "
          f"test={rows[-1]['test_acc']:.4f} cv={cvs.mean():.4f} "
          f"gap={rows[-1]['train_acc']-rows[-1]['test_acc']:+.4f}")

dt_df = pd.DataFrame(rows)
dt_df.to_csv(f"{OUTDIR}/decision_tree_results.csv", index=False)
print("\n" + dt_df.round(4).to_string(index=False))

best_row   = dt_df.loc[dt_df["cv_acc"].idxmax()]
best_depth = None if best_row["max_depth"] == "None" else int(best_row["max_depth"])
print(f"\nBest max_depth by 5-fold CV: {best_row['max_depth']} (cv_acc={best_row['cv_acc']:.4f})")

                                    
xl = [str(d) for d in dt_df["max_depth"]]
plt.figure(figsize=(9, 5))
plt.plot(xl, dt_df["train_acc"], "o-", label="Training accuracy")
plt.plot(xl, dt_df["test_acc"],  "s-", label="Test accuracy")
plt.plot(xl, dt_df["cv_acc"],    "^--", label="5-fold CV accuracy")
plt.xlabel("max_depth"); plt.ylabel("Accuracy")
plt.title("Decision Tree: effect of max_depth on accuracy")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig1_dt_depth_curve.png", bbox_inches="tight"); plt.show()

                                    
x = np.arange(len(dt_df)); w = 0.27
plt.figure(figsize=(10, 5))
plt.bar(x-w, dt_df["train_acc"], w, label="Train")
plt.bar(x,   dt_df["test_acc"],  w, label="Test")
plt.bar(x+w, dt_df["cv_acc"],    w, label="5-fold CV")
plt.xticks(x, xl); plt.ylim(0.6, 1.02)
plt.xlabel("max_depth"); plt.ylabel("Accuracy")
plt.title("Decision Tree accuracy at different maximum depths")
plt.legend(); plt.grid(axis="y", alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig2_dt_bar.png", bbox_inches="tight"); plt.show()

                                       
plt.figure(figsize=(16, 8))
plot_tree(trees[best_depth], max_depth=3, feature_names=FEATURES,
          class_names=["Died", "Survived"], filled=True, rounded=True, fontsize=8)
plt.title(f"Decision Tree (max_depth={best_row['max_depth']}) - top 3 levels shown")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/fig3_tree.png", bbox_inches="tight"); plt.show()

                                     
imp = pd.Series(trees[best_depth].feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(7, 4))
plt.barh(imp.index, imp.values)
plt.xlabel("Gini importance")
plt.title(f"Feature importance (max_depth={best_row['max_depth']})")
plt.grid(axis="x", alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig4_importance.png", bbox_inches="tight"); plt.show()

                                                                              
                                                    
                                                                              
print("\n" + "="*70)
print("PART B - NAIVE BAYES (5-fold cross-validation)")
print("="*70)

nb_scores = cross_val_score(GaussianNB(), X, y, cv=cv, scoring="accuracy")
for i, s in enumerate(nb_scores, 1):
    print(f"Fold {i}: accuracy = {s:.4f}")
print(f"\nAverage accuracy = {nb_scores.mean():.4f}  (std = {nb_scores.std():.4f})")

nb_df = pd.DataFrame({"fold": [f"Fold {i}" for i in range(1, 6)] + ["Average"],
                      "accuracy": list(nb_scores) + [nb_scores.mean()]})
nb_df.to_csv(f"{OUTDIR}/naive_bayes_cv.csv", index=False)

                                        
plt.figure(figsize=(8, 5))
cols = ["#4C72B0"]*5 + ["#DD8452"]
bars = plt.bar(nb_df["fold"], nb_df["accuracy"], color=cols)
plt.axhline(nb_scores.mean(), ls="--", color="red", lw=1,
            label=f"Mean = {nb_scores.mean():.4f}")
for b, v in zip(bars, nb_df["accuracy"]):
    plt.text(b.get_x()+b.get_width()/2, v+0.006, f"{v:.4f}", ha="center", fontsize=9)
plt.ylim(0, 1.05); plt.ylabel("Accuracy")
plt.title("Naive Bayes - 5-fold cross-validation accuracy")
plt.legend(); plt.grid(axis="y", alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig5_nb_folds.png", bbox_inches="tight"); plt.show()

                                                                              
                      
                                                                              
dt_cv = cross_val_score(DecisionTreeClassifier(max_depth=best_depth,
                        random_state=RANDOM_STATE), X, y, cv=cv, scoring="accuracy")
comp = pd.DataFrame({
    "Classifier": [f"Decision Tree (depth={best_row['max_depth']})", "Naive Bayes (GaussianNB)"],
    "Mean CV accuracy": [dt_cv.mean(), nb_scores.mean()],
    "Std": [dt_cv.std(), nb_scores.std()]})
comp.to_csv(f"{OUTDIR}/comparison.csv", index=False)
print("\n" + comp.round(4).to_string(index=False))

plt.figure(figsize=(7, 5))
bars = plt.bar(comp["Classifier"], comp["Mean CV accuracy"],
               yerr=comp["Std"], capsize=6, color=["#4C72B0", "#DD8452"])
for b, v in zip(bars, comp["Mean CV accuracy"]):
    plt.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.4f}", ha="center", fontsize=10)
plt.ylim(0, 1.0); plt.ylabel("Mean 5-fold CV accuracy")
plt.title("Decision Tree vs Naive Bayes")
plt.grid(axis="y", alpha=.3); plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig6_comparison.png", bbox_inches="tight"); plt.show()

                                     
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, (name, model) in zip(ax, [("Decision Tree", trees[best_depth]),
                                 ("Naive Bayes", GaussianNB().fit(X_train, y_train))]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Died", "Survived"]).plot(ax=a, colorbar=False)
    a.set_title(f"{name}  (test accuracy = {accuracy_score(y_test, model.predict(X_test)):.4f})")
plt.tight_layout(); plt.savefig(f"{OUTDIR}/fig7_confusion.png", bbox_inches="tight"); plt.show()

                                                                              
                                                             
                                                                              
try:
    td = pd.read_csv(TEST_PATH)
    pid = td["PassengerId"]
    Xt = preprocess(td).reindex(columns=FEATURES, fill_value=0)
    out = pd.DataFrame({"PassengerId": pid,
                        "Survived": trees[best_depth].fit(X, y).predict(Xt)})
    out.to_csv(f"{OUTDIR}/test_predictions.csv", index=False)
    print(f"\ntest.csv predictions written ({len(out)} rows). "
          "Note: test.csv has no Survived column, so accuracy cannot be measured on it.")
except Exception as e:
    print("Skipped test.csv predictions:", e)

print("\nSaved to", OUTDIR, ":", sorted(os.listdir(OUTDIR)))